In [ ]:
# =============================================================================
# Prediction of MCI Conversion in Parkinson's Disease
# Using α-Synuclein Biomarkers
#
# Complete Analysis Pipeline: Data Cleaning → Survival Analysis → Figures
# =============================================================================
#
# Author: [Your Name]
# Date: [Date]
#
# Description:
#   This script performs complete analysis pipeline from data cleaning to
#   machine learning-based survival analysis for predicting progression from
#   normal cognition (NC) to mild cognitive impairment (MCI) in PD patients.
#
# Input:
#   - 01062026-longitudinal-DLS_new.csv: Raw longitudinal biomarker data
#
# Output:
#   - Cleaned datasets (step3, step4, step5)
#   - Tables (Univariate Cox, Multivariate Cox, ML combinations, SHAP)
#   - Figures (C-index trajectory, Forest plot, SHAP plots)
#
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
import shap
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("MCI PREDICTION ANALYSIS - COMPLETE PIPELINE")
print("=" * 70)

# =============================================================================
# PART 1: DATA CLEANING
# =============================================================================

print("\n" + "=" * 70)
print("PART 1: DATA CLEANING")
print("=" * 70)

# -----------------------------------------------------------------------------
# Step 1: Load Raw Data
# -----------------------------------------------------------------------------
print("\n[Step 1] Loading raw data...")

# Upload file first in Colab, or specify path
# from google.colab import files
# uploaded = files.upload()

raw_df = pd.read_csv('01062026-longitudinal-DLS_new.csv')
print(f"Raw data: {len(raw_df)} rows")

# Remove empty rows
df = raw_df.dropna(how='all').copy()
print(f"After removing empty rows: {len(df)} rows")

# -----------------------------------------------------------------------------
# Step 2: Standardize Column Names
# -----------------------------------------------------------------------------
print("\n[Step 2] Standardizing column names...")

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '-')
print(f"Columns: {list(df.columns)}")

# -----------------------------------------------------------------------------
# Step 3: Fix Data Quality Issues
# -----------------------------------------------------------------------------
print("\n[Step 3] Fixing data quality issues...")

# 3a. Fix IC patient (use Visit_num for correct visit numbers)
ic_mask = df['patient_id'] == 'IC'
if ic_mask.sum() > 0:
    df.loc[ic_mask, 'visit'] = df.loc[ic_mask, 'visit_num']
    df.loc[ic_mask, 'visit-month'] = (df.loc[ic_mask, 'visit_num'] - 1) * 6
    print(f"  Fixed IC patient: {ic_mask.sum()} rows")

# 3b. Merge FQ patients (FQ, FQ_v3, FQ_v5, FQ_v7, FQ_v9 → FQ)
fq_variants = ['FQ', 'FQ_v3', 'FQ_v5', 'FQ_v7', 'FQ_v9']
fq_mask = df['patient_id'].isin(fq_variants)
if fq_mask.sum() > 0:
    df.loc[fq_mask, 'patient_id'] = 'FQ'
    print(f"  Merged FQ variants: {fq_mask.sum()} rows → FQ")

# 3c. Remove AV duplicate (row with Visit_num=9 that duplicates Visit_num=5)
av_mask = (df['patient_id'] == 'AV') & (df['visit_num'] == 9)
if av_mask.sum() > 0:
    df = df[~av_mask].copy()
    print(f"  Removed AV duplicate: 1 row")

# 3d. Fix AV missing visit-month values
av_mask = df['patient_id'] == 'AV'
if av_mask.sum() > 0:
    df.loc[av_mask, 'visit-month'] = (df.loc[av_mask, 'visit'] - 1) * 6
    print(f"  Fixed AV visit-month: {av_mask.sum()} rows")

print(f"\nCleaned data: {len(df)} rows, {df['patient_id'].nunique()} patients")

# Save Step 3
df.to_csv('step3_data_fixed.csv', index=False)
print("Saved: step3_data_fixed.csv")

# -----------------------------------------------------------------------------
# Step 4: Select Analysis Cohort
# -----------------------------------------------------------------------------
print("\n[Step 4] Selecting analysis cohort...")

# Criteria: NC baseline + ≥2 visits + no reversion

# Get baseline cognitive status for each patient
baseline_status = df.groupby('patient_id').first()['cognitive-status'].reset_index()
baseline_status.columns = ['patient_id', 'baseline_status']

# Get visit counts
visit_counts = df.groupby('patient_id').size().reset_index(name='n_visits')

# Merge
patient_info = pd.merge(baseline_status, visit_counts, on='patient_id')

# Check for reversion (MCI → NC)
def check_reversion(group):
    statuses = group['cognitive-status'].tolist()
    for i in range(1, len(statuses)):
        if statuses[i-1] in ['MCI', 'D'] and statuses[i] == 'NC':
            return True
    return False

reversion_check = df.groupby('patient_id').apply(check_reversion).reset_index()
reversion_check.columns = ['patient_id', 'has_reversion']
patient_info = pd.merge(patient_info, reversion_check, on='patient_id')

# Apply filters
nc_baseline = patient_info[patient_info['baseline_status'] == 'NC']
multi_visit = nc_baseline[nc_baseline['n_visits'] >= 2]
no_reversion = multi_visit[multi_visit['has_reversion'] == False]

selected_patients = no_reversion['patient_id'].tolist()
print(f"  NC baseline: {len(nc_baseline)} patients")
print(f"  ≥2 visits: {len(multi_visit)} patients")
print(f"  No reversion: {len(no_reversion)} patients")
print(f"  Final cohort: {len(selected_patients)} patients")

# Filter data
cohort_df = df[df['patient_id'].isin(selected_patients)].copy()

# Determine converter status
def get_conversion_status(group):
    statuses = group['cognitive-status'].tolist()
    if 'MCI' in statuses or 'D' in statuses:
        return 'Converter'
    return 'Stable'

conversion_status = cohort_df.groupby('patient_id').apply(get_conversion_status).reset_index()
conversion_status.columns = ['patient_id', 'conversion_status']

n_converters = (conversion_status['conversion_status'] == 'Converter').sum()
n_stable = (conversion_status['conversion_status'] == 'Stable').sum()
print(f"  Converters: {n_converters}, Stable: {n_stable}")

# Save Step 4
cohort_df.to_csv('step4_analysis_cohort.csv', index=False)
conversion_status.to_csv('step4_patient_info.csv', index=False)
print("Saved: step4_analysis_cohort.csv, step4_patient_info.csv")

# -----------------------------------------------------------------------------
# Step 5: Create Counting Process Format
# -----------------------------------------------------------------------------
print("\n[Step 5] Creating counting process format...")

# Define biomarker columns
biomarker_cols = [
    'dls-peak-number', 'dls-peak-1-intensity', 'dls-peak-2-intensity',
    'dls-peak-1-size', 'dls-peak-2-size', 'dls-peak-1-pd', 'dls-peak-2-pd',
    'tht-mfi', 'tht-tlag', 'tht-t20', 'tht-t50', 'tht-forming-rate',
    'neurotoxicity', 'age', 'sex'
]

# Check available columns
available_biomarkers = [col for col in biomarker_cols if col in cohort_df.columns]
print(f"  Available biomarkers: {len(available_biomarkers)}")

counting_process_rows = []

for patient_id in selected_patients:
    patient_data = cohort_df[cohort_df['patient_id'] == patient_id].sort_values('visit-month')
    visits = patient_data['visit-month'].tolist()
    statuses = patient_data['cognitive-status'].tolist()

    # Find event time (first MCI/D)
    event_time = None
    for i, status in enumerate(statuses):
        if status in ['MCI', 'D']:
            event_time = visits[i]
            break

    # Create intervals
    for i in range(len(visits)):
        if i == 0:
            start = 0
        else:
            start = visits[i-1] if visits[i-1] < visits[i] else visits[i]

        stop = visits[i]

        # Skip if start >= stop
        if start >= stop:
            if i == 0:
                start = 0
                stop = visits[i] if visits[i] > 0 else 12
            else:
                continue

        # Determine event
        if event_time is not None and stop == event_time:
            event = 1
        else:
            event = 0

        # Get biomarker values at this visit
        visit_data = patient_data[patient_data['visit-month'] == visits[i]]
        if len(visit_data) == 0:
            continue

        row = {
            'patient_id': patient_id,
            'start': start,
            'stop': stop,
            'event': event
        }

        for col in available_biomarkers:
            if col in visit_data.columns:
                row[col] = visit_data[col].values[0]

        counting_process_rows.append(row)

counting_df = pd.DataFrame(counting_process_rows)

# Remove rows where start >= stop
counting_df = counting_df[counting_df['start'] < counting_df['stop']].copy()

print(f"  Counting process data: {len(counting_df)} intervals")
print(f"  Patients: {counting_df['patient_id'].nunique()}")
print(f"  Events: {counting_df['event'].sum()}")

# Save Step 5
counting_df.to_csv('step5_counting_process.csv', index=False)
print("Saved: step5_counting_process.csv")

# =============================================================================
# PART 2: MACHINE LEARNING SURVIVAL ANALYSIS
# =============================================================================

print("\n" + "=" * 70)
print("PART 2: MACHINE LEARNING SURVIVAL ANALYSIS")
print("=" * 70)

# Load counting process data
df = pd.read_csv('step5_counting_process.csv')
df.columns = df.columns.str.replace('.', '-', regex=False)

print(f"\n[Data Summary]")
print(f"Total intervals: {len(df)}")
print(f"Patients: {df['patient_id'].nunique()}")
print(f"Events: {df['event'].sum()}")

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
CORE_VARS = ['dls-peak-number', 'sex']
ADDITIONAL_VARS = [
    'dls-peak-2-pd', 'dls-peak-2-size', 'dls-peak-2-intensity',
    'dls-peak-1-intensity', 'dls-peak-1-size',
    'tht-mfi', 'tht-t50', 'tht-t20'
]
N_SPLITS = 5
N_ITERATIONS = 10

print(f"\n[Configuration]")
print(f"Core variables: {CORE_VARS}")
print(f"CV: {N_SPLITS}-fold × {N_ITERATIONS} iterations")

# -----------------------------------------------------------------------------
# Model and CV Functions
# -----------------------------------------------------------------------------
def get_models():
    return {
        'LASSO-Cox': CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.1, max_iter=1000),
        'RSF': RandomSurvivalForest(n_estimators=100, max_depth=3, random_state=None, n_jobs=-1),
        'GBM': GradientBoostingSurvivalAnalysis(n_estimators=100, max_depth=2,
                                                 learning_rate=0.1, random_state=None)
    }

def prepare_data(df, variables):
    X = df[variables].values
    y_time = df['stop'].values
    y_event = df['event'].astype(bool).values
    groups = df['patient_id'].values
    y = np.array([(e, t) for e, t in zip(y_event, y_time)],
                 dtype=[('event', bool), ('time', float)])
    return X, y, y_event, groups

def run_cv_with_iterations(df, variables, n_splits=5, n_iterations=10):
    X, y, y_event, groups = prepare_data(df, variables)
    all_results = {name: [] for name in ['LASSO-Cox', 'RSF', 'GBM']}

    for iteration in range(n_iterations):
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42 + iteration)
        for fold, (train_idx, test_idx) in enumerate(cv.split(X, y_event, groups)):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            if y_test['event'].sum() == 0:
                continue

            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            for model_name in ['LASSO-Cox', 'RSF', 'GBM']:
                try:
                    model = get_models()[model_name]
                    model.fit(X_train_scaled, y_train)
                    pred = model.predict(X_test_scaled)
                    c_idx = concordance_index_censored(y_test['event'], y_test['time'], pred)[0]
                    all_results[model_name].append(c_idx)
                except:
                    continue

    summary = {}
    for model_name, scores in all_results.items():
        if len(scores) > 0:
            summary[model_name] = {'mean': np.mean(scores), 'std': np.std(scores)}
    return summary

# -----------------------------------------------------------------------------
# 2-Variable (Core Model)
# -----------------------------------------------------------------------------
print("\n" + "-" * 50)
print("2-Variable Model (Core): dls-peak-number + sex")
print("-" * 50)

core_results = run_cv_with_iterations(df, CORE_VARS, N_SPLITS, N_ITERATIONS)
for model, res in core_results.items():
    print(f"  {model}: {res['mean']:.3f} ± {res['std']:.3f}")
avg_2var = np.mean([r['mean'] for r in core_results.values()])
print(f"  Average: {avg_2var:.3f}")

# -----------------------------------------------------------------------------
# 3-Variable Combinations
# -----------------------------------------------------------------------------
print("\n" + "-" * 50)
print("3-Variable Combinations (Core + 1)")
print("-" * 50)

three_var_results = []
for add_var in ADDITIONAL_VARS:
    test_vars = CORE_VARS + [add_var]
    results = run_cv_with_iterations(df, test_vars, N_SPLITS, N_ITERATIONS)
    avg_c = np.mean([r['mean'] for r in results.values()])

    three_var_results.append({
        'Added_Variable': add_var,
        'Variables': ' + '.join(test_vars),
        'N_vars': 3,
        'LASSO_Cox': f"{results['LASSO-Cox']['mean']:.3f} ± {results['LASSO-Cox']['std']:.3f}",
        'RSF': f"{results['RSF']['mean']:.3f} ± {results['RSF']['std']:.3f}",
        'GBM': f"{results['GBM']['mean']:.3f} ± {results['GBM']['std']:.3f}",
        'Avg_C-index': round(avg_c, 3),
        'Change': round(avg_c - avg_2var, 3)
    })
    print(f"  + {add_var}: {avg_c:.3f} (Δ = {avg_c - avg_2var:+.3f})")

three_var_df = pd.DataFrame(three_var_results).sort_values('Avg_C-index', ascending=False)
best_3rd = three_var_df.iloc[0]['Added_Variable']
best_3_vars = CORE_VARS + [best_3rd]
avg_3var = three_var_df.iloc[0]['Avg_C-index']
print(f"\n  Best 3-var: {best_3_vars}, C-index: {avg_3var}")

# -----------------------------------------------------------------------------
# 4-Variable Combinations
# -----------------------------------------------------------------------------
print("\n" + "-" * 50)
print(f"4-Variable Combinations (Best 3 + 1)")
print("-" * 50)

remaining_vars = [v for v in ADDITIONAL_VARS if v != best_3rd]
four_var_results = []

for add_var in remaining_vars:
    test_vars = best_3_vars + [add_var]
    results = run_cv_with_iterations(df, test_vars, N_SPLITS, N_ITERATIONS)
    avg_c = np.mean([r['mean'] for r in results.values()])

    four_var_results.append({
        'Added_Variable': add_var,
        'Variables': ' + '.join(test_vars),
        'N_vars': 4,
        'LASSO_Cox': f"{results['LASSO-Cox']['mean']:.3f} ± {results['LASSO-Cox']['std']:.3f}",
        'RSF': f"{results['RSF']['mean']:.3f} ± {results['RSF']['std']:.3f}",
        'GBM': f"{results['GBM']['mean']:.3f} ± {results['GBM']['std']:.3f}",
        'Avg_C-index': round(avg_c, 3),
        'Change': round(avg_c - avg_3var, 3)
    })
    print(f"  + {add_var}: {avg_c:.3f} (Δ = {avg_c - avg_3var:+.3f})")

four_var_df = pd.DataFrame(four_var_results).sort_values('Avg_C-index', ascending=False)
best_4th = four_var_df.iloc[0]['Added_Variable']
best_4_vars = best_3_vars + [best_4th]
avg_4var = four_var_df.iloc[0]['Avg_C-index']
print(f"\n  Best 4-var: {best_4_vars}, C-index: {avg_4var}")

# -----------------------------------------------------------------------------
# 5-Variable Combinations (Overfitting Check)
# -----------------------------------------------------------------------------
print("\n" + "-" * 50)
print(f"5-Variable Combinations (Best 4 + 1) - Overfitting Check")
print("-" * 50)

remaining_vars_2 = [v for v in remaining_vars if v != best_4th]
five_var_results = []

for add_var in remaining_vars_2:
    test_vars = best_4_vars + [add_var]
    results = run_cv_with_iterations(df, test_vars, N_SPLITS, N_ITERATIONS)
    avg_c = np.mean([r['mean'] for r in results.values()])

    five_var_results.append({
        'Added_Variable': add_var,
        'Variables': ' + '.join(test_vars),
        'N_vars': 5,
        'LASSO_Cox': f"{results['LASSO-Cox']['mean']:.3f} ± {results['LASSO-Cox']['std']:.3f}",
        'RSF': f"{results['RSF']['mean']:.3f} ± {results['RSF']['std']:.3f}",
        'GBM': f"{results['GBM']['mean']:.3f} ± {results['GBM']['std']:.3f}",
        'Avg_C-index': round(avg_c, 3),
        'Change': round(avg_c - avg_4var, 3)
    })
    print(f"  + {add_var}: {avg_c:.3f} (Δ = {avg_c - avg_4var:+.3f})")

five_var_df = pd.DataFrame(five_var_results).sort_values('Avg_C-index', ascending=False)
avg_5var = five_var_df.iloc[0]['Avg_C-index']
best_5_vars = best_4_vars + [five_var_df.iloc[0]['Added_Variable']]

# -----------------------------------------------------------------------------
# Trajectory Summary
# -----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("C-INDEX TRAJECTORY")
print("=" * 70)

trajectory_data = [
    {'N_vars': 2, 'Variables': ' + '.join(CORE_VARS), 'C-index': round(avg_2var, 3), 'Change': '-'},
    {'N_vars': 3, 'Variables': ' + '.join(best_3_vars), 'C-index': round(avg_3var, 3), 'Change': f"{avg_3var - avg_2var:+.3f}"},
    {'N_vars': 4, 'Variables': ' + '.join(best_4_vars), 'C-index': round(avg_4var, 3), 'Change': f"{avg_4var - avg_3var:+.3f}"},
    {'N_vars': 5, 'Variables': ' + '.join(best_5_vars), 'C-index': round(avg_5var, 3), 'Change': f"{avg_5var - avg_4var:+.3f}"}
]
trajectory_df = pd.DataFrame(trajectory_data)
print(trajectory_df.to_string(index=False))

# Determine optimal
optimal_idx = trajectory_df['C-index'].idxmax()
optimal_n_vars = trajectory_df.loc[optimal_idx, 'N_vars']
optimal_c = trajectory_df.loc[optimal_idx, 'C-index']

if optimal_n_vars == 2:
    OPTIMAL_VARS = CORE_VARS
elif optimal_n_vars == 3:
    OPTIMAL_VARS = best_3_vars
elif optimal_n_vars == 4:
    OPTIMAL_VARS = best_4_vars
else:
    OPTIMAL_VARS = best_5_vars

print(f"\n*** OPTIMAL: {optimal_n_vars} variables, C-index = {optimal_c}")
print(f"*** Variables: {OPTIMAL_VARS}")

# =============================================================================
# PART 3: SHAP ANALYSIS
# =============================================================================

print("\n" + "=" * 70)
print("PART 3: SHAP ANALYSIS")
print("=" * 70)

X, y, y_event, groups = prepare_data(df, OPTIMAL_VARS)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train models
rsf = RandomSurvivalForest(n_estimators=100, max_depth=3, random_state=42, n_jobs=-1)
rsf.fit(X_scaled, y)

gbm = GradientBoostingSurvivalAnalysis(n_estimators=100, max_depth=2, learning_rate=0.1, random_state=42)
gbm.fit(X_scaled, y)

# SHAP values
print("Calculating SHAP values...")
explainer_rsf = shap.Explainer(rsf.predict, X_scaled, feature_names=OPTIMAL_VARS)
shap_values_rsf = explainer_rsf(X_scaled)

explainer_gbm = shap.Explainer(gbm.predict, X_scaled, feature_names=OPTIMAL_VARS)
shap_values_gbm = explainer_gbm(X_scaled)

# Feature importance
shap_importance = pd.DataFrame({
    'Variable': OPTIMAL_VARS,
    'RSF_SHAP': np.abs(shap_values_rsf.values).mean(axis=0),
    'GBM_SHAP': np.abs(shap_values_gbm.values).mean(axis=0)
})
shap_importance['Average_SHAP'] = (shap_importance['RSF_SHAP'] + shap_importance['GBM_SHAP']) / 2
shap_importance = shap_importance.sort_values('Average_SHAP', ascending=False)

print("\n[SHAP Feature Importance]")
print(shap_importance.round(4).to_string(index=False))

# =============================================================================
# PART 4: GENERATE FIGURES
# =============================================================================

print("\n" + "=" * 70)
print("PART 4: GENERATING FIGURES")
print("=" * 70)

with PdfPages('Final_Figures_All.pdf') as pdf:

    # Figure 1: C-index Trajectory
    fig, ax = plt.subplots(figsize=(10, 6))
    n_vars_list = trajectory_df['N_vars'].values
    c_index_list = trajectory_df['C-index'].values

    ax.plot(n_vars_list, c_index_list, 'o-', linewidth=2.5, markersize=14,
            color='steelblue', markerfacecolor='white', markeredgewidth=2.5)
    ax.scatter([optimal_n_vars], [optimal_c], s=250, color='red', zorder=5, marker='*')

    for i, row in trajectory_df.iterrows():
        ax.annotate(f"C={row['C-index']}", (row['N_vars'], row['C-index'] + 0.012),
                    ha='center', fontsize=11, fontweight='bold' if row['N_vars'] == optimal_n_vars else 'normal')

    ax.set_xlabel('Number of Variables', fontsize=13)
    ax.set_ylabel('C-index (5-Fold × 10 Iterations CV)', fontsize=13)
    ax.set_title('Figure 1: Model Performance vs. Number of Variables', fontsize=14, fontweight='bold')
    ax.set_xticks(n_vars_list)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=optimal_c, color='red', linestyle='--', alpha=0.3)

    plt.tight_layout()
    pdf.savefig(bbox_inches='tight')
    plt.savefig('Figure1_Cindex_Trajectory.pdf', format='pdf', bbox_inches='tight')
    plt.close()
    print("Saved: Figure1_Cindex_Trajectory.pdf")

    # Figure 2: Forest Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    forest_data = {
        'Variable': ['DLS peak number', 'DLS peak 2 PD', 'DLS peak 2 intensity',
                     'Sex (male)', 'DLS peak 1 intensity', 'DLS peak 2 size'],
        'HR': [0.12, 0.91, 0.73, 4.56, 1.07, 0.98],
        'CI_lower': [0.03, 0.85, 0.59, 1.34, 1.01, 0.97],
        'CI_upper': [0.45, 0.97, 0.92, 15.48, 1.13, 1.00],
        'P_value': [0.002, 0.005, 0.006, 0.015, 0.025, 0.046]
    }
    forest_df = pd.DataFrame(forest_data)

    for i, row in forest_df.iterrows():
        color = 'steelblue' if row['HR'] < 1 else 'darkorange'
        ax.errorbar(row['HR'], i, xerr=[[row['HR'] - row['CI_lower']], [row['CI_upper'] - row['HR']]],
                    fmt='o', markersize=10, color=color, capsize=5, capthick=2, linewidth=2)

    ax.axvline(x=1, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.set_yticks(range(len(forest_df)))
    ax.set_yticklabels(forest_df['Variable'], fontsize=11)
    ax.set_xlabel('Hazard Ratio (95% CI)', fontsize=13)
    ax.set_title('Figure 2: Univariate Cox Regression - Hazard Ratios', fontsize=14, fontweight='bold')
    ax.set_xscale('log')
    ax.set_xlim([0.01, 25])
    ax.grid(axis='x', alpha=0.3)

    for i, row in forest_df.iterrows():
        ax.annotate(f"p={row['P_value']:.3f}", (row['CI_upper'] * 1.4, i), fontsize=10, va='center')

    plt.tight_layout()
    pdf.savefig(bbox_inches='tight')
    plt.savefig('Figure2_Forest_Plot.pdf', format='pdf', bbox_inches='tight')
    plt.close()
    print("Saved: Figure2_Forest_Plot.pdf")

    # Figure 3a: SHAP RSF
    fig = plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values_rsf, X_scaled, feature_names=OPTIMAL_VARS, show=False)
    plt.title('Figure 3a: SHAP Summary - Random Survival Forest', fontsize=14, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(bbox_inches='tight')
    plt.savefig('Figure3a_SHAP_RSF.pdf', format='pdf', bbox_inches='tight')
    plt.close()
    print("Saved: Figure3a_SHAP_RSF.pdf")

    # Figure 3b: SHAP GBM
    fig = plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values_gbm, X_scaled, feature_names=OPTIMAL_VARS, show=False)
    plt.title('Figure 3b: SHAP Summary - Gradient Boosting Machine', fontsize=14, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(bbox_inches='tight')
    plt.savefig('Figure3b_SHAP_GBM.pdf', format='pdf', bbox_inches='tight')
    plt.close()
    print("Saved: Figure3b_SHAP_GBM.pdf")

    # Figure 4: SHAP Bar
    fig, ax = plt.subplots(figsize=(9, 5))
    sorted_imp = shap_importance.sort_values('Average_SHAP', ascending=True)
    x = np.arange(len(OPTIMAL_VARS))
    width = 0.35

    ax.barh(x - width/2, sorted_imp['RSF_SHAP'], width, label='RSF', color='steelblue')
    ax.barh(x + width/2, sorted_imp['GBM_SHAP'], width, label='GBM', color='darkorange')

    ax.set_xlabel('Mean |SHAP value|', fontsize=12)
    ax.set_ylabel('Variable', fontsize=12)
    ax.set_title('Figure 4: Feature Importance - SHAP Values', fontsize=14, fontweight='bold')
    ax.set_yticks(x)
    ax.set_yticklabels(sorted_imp['Variable'], fontsize=11)
    ax.legend()
    ax.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    pdf.savefig(bbox_inches='tight')
    plt.savefig('Figure4_SHAP_Bar.pdf', format='pdf', bbox_inches='tight')
    plt.close()
    print("Saved: Figure4_SHAP_Bar.pdf")

print("Saved: Final_Figures_All.pdf")

# =============================================================================
# PART 5: SAVE TABLES
# =============================================================================

print("\n" + "=" * 70)
print("PART 5: SAVING TABLES")
print("=" * 70)

three_var_df.to_csv('Table_3var_Combinations.csv', index=False)
four_var_df.to_csv('Table_4var_Combinations.csv', index=False)
five_var_df.to_csv('Table_5var_Combinations.csv', index=False)
trajectory_df.to_csv('Table_Cindex_Trajectory.csv', index=False)
shap_importance.to_csv('Table_SHAP_Importance.csv', index=False)

print("Saved: Table_3var_Combinations.csv")
print("Saved: Table_4var_Combinations.csv")
print("Saved: Table_5var_Combinations.csv")
print("Saved: Table_Cindex_Trajectory.csv")
print("Saved: Table_SHAP_Importance.csv")

# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)

print(f"""
KEY FINDINGS:
  - Core model (2 vars): C-index = {avg_2var:.3f}
  - Optimal model ({optimal_n_vars} vars): C-index = {optimal_c:.3f}
  - Variables: {', '.join(OPTIMAL_VARS)}
  - Overfitting detected at 5 variables (C-index decreases)

CV SETTINGS: {N_SPLITS}-fold × {N_ITERATIONS} iterations
""")

In [ ]:
# =============================================================================
# Prediction of MCI Conversion in Parkinson's Disease
# Using Alpha-Synuclein Biomarkers - Longitudinal Survival Analysis
#
# Description:
#   Complete analysis pipeline for predicting progression from normal
#   cognition (NC) to mild cognitive impairment (MCI) in PD patients.
#   Based on time-varying Cox regression and ML survival models.
#
# Input:
#   - step5_counting_process.csv : Previous counting process file
#   - 02092026_longitudinal.csv  : Raw longitudinal data (with education,
#                                  disease duration)
#
# Output:
#   - counting_process_final_v2.csv
#   - Univariate_Cox_Results_v2.xlsx
#   - Figure_Forest_Plot_v2.pdf / .png
#   - Forward_Selection_Complete.xlsx
#   - Top3_Foldwise_Results.xlsx
#   - SHAP_Results_Top3.xlsx
#   - SHAP_Analysis_Top3.pdf
#
# CV Strategy:
#   StratifiedGroupKFold, 5-fold x 10 iterations, interval-level data
#
# =============================================================================

# -----------------------------------------------------------------------------
# Install dependencies (run once)
# -----------------------------------------------------------------------------
# !pip install scikit-survival shap openpyxl lifelines

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from lifelines import CoxTimeVaryingFitter
import shap
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("MCI PREDICTION - LONGITUDINAL SURVIVAL ANALYSIS PIPELINE")
print("=" * 70)

# =============================================================================
# PART 1: COUNTING PROCESS FORMAT 재구성
#         (이전 파일 기반 + education, disease duration 추가)
# =============================================================================

print("\n" + "=" * 70)
print("PART 1: COUNTING PROCESS FORMAT")
print("=" * 70)

# -----------------------------------------------------------------------------
# Step 1: 파일 로드
# -----------------------------------------------------------------------------
print("\n[Step 1] 파일 로드...")

df_old = pd.read_csv('/content/step5_counting_process.csv')
df_raw = pd.read_csv('/content/02092026_longitudinal.csv')

print(f"  이전 CP 파일:  {df_old.shape[0]}행 x {df_old.shape[1]}열")
print(f"  원본 데이터:   {df_raw.shape[0]}행 x {df_raw.shape[1]}열")

# -----------------------------------------------------------------------------
# Step 2: 대상 환자 필터링
# -----------------------------------------------------------------------------
print("\n[Step 2] 대상 환자 필터링...")

target_patients = sorted(df_old['patient_id'].unique().tolist())
df_raw_filtered = df_raw[df_raw['patient_id'].isin(target_patients)].copy()
df_raw_filtered = df_raw_filtered.sort_values(['patient_id', 'visit-month'])

print(f"  대상 환자: {len(target_patients)}명")

# -----------------------------------------------------------------------------
# Step 3: IC visit-month 재정의 + AV 중복 제거
# -----------------------------------------------------------------------------
print("\n[Step 3] IC 재정의 및 AV 중복 제거...")

ic_idx = df_raw_filtered[df_raw_filtered['patient_id'] == 'IC'].index
min_month_ic = df_raw_filtered.loc[ic_idx, 'visit-month'].min()
if min_month_ic != 0:
    df_raw_filtered.loc[ic_idx, 'visit-month'] -= min_month_ic
    print(f"  IC visit-month 재정의: -{min_month_ic}")

before = len(df_raw_filtered)
df_raw_filtered = df_raw_filtered.drop_duplicates(
    subset=['patient_id', 'visit-month'], keep='first')
print(f"  AV 중복 제거: {before}행 -> {len(df_raw_filtered)}행")

# -----------------------------------------------------------------------------
# Step 4: start 기준으로 education, disease duration 매핑
# -----------------------------------------------------------------------------
print("\n[Step 4] education, disease duration 매핑...")

mapping = {}
for _, row in df_raw_filtered.iterrows():
    key = (row['patient_id'], row['visit-month'])
    mapping[key] = {
        'education':        row['education'],
        'disease duration': row['disease duration']
    }

df_cp = df_old.copy()
df_cp['education']        = np.nan
df_cp['disease duration'] = np.nan

matched, unmatched = 0, []
for idx, row in df_cp.iterrows():
    key = (row['patient_id'], row['start'])
    if key in mapping:
        df_cp.loc[idx, 'education']        = mapping[key]['education']
        df_cp.loc[idx, 'disease duration'] = mapping[key]['disease duration']
        matched += 1
    else:
        unmatched.append((row['patient_id'], row['start']))

print(f"  매핑 성공: {matched}개 / 실패: {len(unmatched)}개")

# -----------------------------------------------------------------------------
# Step 5: 검증 및 저장
# -----------------------------------------------------------------------------
print("\n[Step 5] 검증...")

assert len(df_cp) == len(df_old),            "행 수 불일치!"
assert df_cp['event'].sum() == 13,           "이벤트 수 불일치!"
assert df_cp['patient_id'].nunique() == 21,  "환자 수 불일치!"
assert df_cp['education'].isnull().sum() == 0, "education 결측치 존재!"

print(f"  Shape:   {df_cp.shape}")
print(f"  환자:    {df_cp['patient_id'].nunique()}명")
print(f"  구간:    {len(df_cp)}개")
print(f"  이벤트:  {int(df_cp['event'].sum())}개")
print(f"  모든 검증 통과!")

df_cp.to_csv('/content/counting_process_final_v2.csv', index=False)
print("  저장: counting_process_final_v2.csv")

# =============================================================================
# PART 2: UNIVARIATE TIME-VARYING COX REGRESSION
# =============================================================================

print("\n" + "=" * 70)
print("PART 2: UNIVARIATE TIME-VARYING COX REGRESSION")
print("=" * 70)

# 변수 정의
VARIABLES = {
    'THT':          ['tht-mfi', 'tht-tlag', 'tht-t20',
                     'tht-t50', 'tht-forming-rate'],
    'DLS':          ['dls-peak-number', 'dls-peak-1-intensity',
                     'dls-peak-2-intensity', 'dls-peak-1-size',
                     'dls-peak-2-size', 'dls-peak-1-pd', 'dls-peak-2-pd'],
    'Neurotoxicity':['neurotoxicity'],
    'Demographics': ['age', 'sex', 'education', 'disease duration']
}
ALL_VARS = [v for vlist in VARIABLES.values() for v in vlist]

def run_univariate_cox(df, variable):
    try:
        safe_name = 'var_x'
        cols = ['patient_id', 'start', 'stop', 'event', variable]
        df_var = df[cols].dropna().copy()
        df_var = df_var.rename(columns={variable: safe_name})

        ctv = CoxTimeVaryingFitter()
        ctv.fit(df_var, id_col='patient_id', event_col='event',
                start_col='start', stop_col='stop')

        s = ctv.summary
        return {
            'Variable':    variable,
            'HR':          round(float(np.exp(s.loc[safe_name, 'coef'])), 4),
            'HR_lower_95': round(float(np.exp(s.loc[safe_name, 'coef lower 95%'])), 4),
            'HR_upper_95': round(float(np.exp(s.loc[safe_name, 'coef upper 95%'])), 4),
            'Z':           round(float(s.loc[safe_name, 'z']), 4),
            'P_value':     round(float(s.loc[safe_name, 'p']), 4),
            'Status':      'OK'
        }
    except Exception as e:
        return {'Variable': variable, 'HR': np.nan, 'HR_lower_95': np.nan,
                'HR_upper_95': np.nan, 'Z': np.nan, 'P_value': np.nan,
                'Status': f'ERROR: {str(e)}'}

print("\n[Univariate Cox 실행]")
uni_results = []
for group_name, var_list in VARIABLES.items():
    print(f"\n  [{group_name}]")
    for var in var_list:
        result = run_univariate_cox(df_cp, var)
        uni_results.append(result)
        if result['Status'] == 'OK':
            p = result['P_value']
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '.' if p < 0.1 else ''
            print(f"    {var:<30} HR={result['HR']:.3f} "
                  f"({result['HR_lower_95']:.3f}-{result['HR_upper_95']:.3f}) "
                  f"p={result['P_value']:.4f} {sig}")

df_uni = pd.DataFrame(uni_results)
df_uni_ok = df_uni[df_uni['Status'] == 'OK'].sort_values('P_value').reset_index(drop=True)

print(f"\n  p<0.05 변수: {df_uni_ok[df_uni_ok['P_value']<0.05]['Variable'].tolist()}")

df_uni_ok.to_excel('/content/Univariate_Cox_Results_v2.xlsx', index=False)
print("  저장: Univariate_Cox_Results_v2.xlsx")

# =============================================================================
# PART 3: FOREST PLOT (p < 0.05 변수)
# =============================================================================

print("\n" + "=" * 70)
print("PART 3: FOREST PLOT")
print("=" * 70)

# p < 0.05 변수만
df_forest = df_uni_ok[df_uni_ok['P_value'] < 0.05].copy()
df_forest = df_forest.sort_values('P_value', ascending=False).reset_index(drop=True)

# 변수명 표시용 매핑
DISPLAY_NAMES = {
    'dls-peak-number':      'DLS peak number',
    'dls-peak-2-pd':        'DLS peak 2 PD',
    'dls-peak-2-intensity': 'DLS peak 2 intensity',
    'sex':                  'Sex (male)',
    'dls-peak-1-intensity': 'DLS peak 1 intensity',
    'education':            'Education',
    'dls-peak-2-size':      'DLS peak 2 size',
}
df_forest['Display_name'] = df_forest['Variable'].map(
    lambda x: DISPLAY_NAMES.get(x, x))

COLOR_MAP = {
    'dls-peak-number':      '#2878B5',
    'dls-peak-2-pd':        '#2878B5',
    'dls-peak-2-intensity': '#2878B5',
    'sex':                  '#FF8C00',
    'dls-peak-1-intensity': '#2878B5',
    'education':            '#FF8C00',
    'dls-peak-2-size':      '#2878B5',
}

fig, ax = plt.subplots(figsize=(10, 6))

for i, row in df_forest.iterrows():
    color  = COLOR_MAP.get(row['Variable'], '#2878B5')
    marker = 's' if row['Variable'] in ['sex', 'education'] else 'o'
    ax.plot([row['HR_lower_95'], row['HR_upper_95']], [i, i],
            color=color, linewidth=1.5, zorder=2)
    ax.scatter(row['HR'], i, color=color, s=80, marker=marker, zorder=3)
    ax.text(row['HR_upper_95'] * 1.15, i,
            f"p={row['P_value']:.3f}", va='center', ha='left', fontsize=10)

ax.axvline(x=1, color='black', linestyle='--', linewidth=1.2, alpha=0.7)
ax.set_yticks(range(len(df_forest)))
ax.set_yticklabels(df_forest['Display_name'].tolist(), fontsize=11)
ax.set_xscale('log')
ax.set_xlabel('Hazard Ratio (95% CI)', fontsize=12)
ax.set_title('Univariate Cox Regression - Hazard Ratios',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlim(0.01, 30)
ax.set_xticks([0.01, 0.1, 1, 10])
ax.set_xticklabels(['$10^{-2}$', '$10^{-1}$', '$10^{0}$', '$10^{1}$'], fontsize=10)
ax.xaxis.grid(True, alpha=0.3)
ax.set_ylim(-0.7, len(df_forest) - 0.3)

legend_elements = [
    mpatches.Patch(color='#2878B5', label='DLS Biomarker'),
    mpatches.Patch(color='#FF8C00', label='Demographics'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.tight_layout()

with PdfPages('/content/Figure_Forest_Plot_v2.pdf') as pdf:
    pdf.savefig(fig, bbox_inches='tight', dpi=300)
fig.savefig('/content/Figure_Forest_Plot_v2.png', bbox_inches='tight',
            dpi=300, facecolor='white')
plt.close()
print("  저장: Figure_Forest_Plot_v2.pdf / .png")

# =============================================================================
# PART 4: ML SURVIVAL ANALYSIS - FORWARD SELECTION
# =============================================================================

print("\n" + "=" * 70)
print("PART 4: ML SURVIVAL ANALYSIS - FORWARD SELECTION")
print("=" * 70)

# -----------------------------------------------------------------------------
# 설정
# -----------------------------------------------------------------------------
CORE_VARS = ['dls-peak-number', 'sex']

ADDITIONAL_VARS = [
    'dls-peak-2-pd',          # univariate p=0.005
    'dls-peak-2-intensity',   # univariate p=0.007
    'dls-peak-1-intensity',   # univariate p=0.025
    'education',              # univariate p=0.025
    'dls-peak-2-size',        # univariate p=0.046
    'dls-peak-1-size',        # univariate p=0.087
    'neurotoxicity',          # univariate p=0.135
    'dls-peak-1-pd',          # univariate p=0.180
    'tht-forming-rate',       # univariate p=0.242
    'age',                    # univariate p=0.368
    'tht-mfi',                # univariate p=0.387
    'disease duration',       # univariate p=0.535
    'tht-tlag',               # univariate p=0.777
    'tht-t20',                # univariate p=0.767
    'tht-t50',                # univariate p=0.752
]

N_SPLITS     = 5
N_ITERATIONS = 10

print(f"  Core:        {CORE_VARS}")
print(f"  Additional:  {len(ADDITIONAL_VARS)}개")
print(f"  CV:          {N_SPLITS}-fold x {N_ITERATIONS} iterations")

# -----------------------------------------------------------------------------
# 함수 정의
# -----------------------------------------------------------------------------
def get_models():
    return {
        'LASSO-Cox': CoxnetSurvivalAnalysis(
            l1_ratio=1.0, alpha_min_ratio=0.1, max_iter=1000),
        'RSF': RandomSurvivalForest(
            n_estimators=100, max_depth=3,
            random_state=None, n_jobs=-1),
        'GBM': GradientBoostingSurvivalAnalysis(
            n_estimators=100, max_depth=2,
            learning_rate=0.1, random_state=None)
    }

def prepare_data(df, variables):
    df_model = df.copy()
    rename_map = {v: v.replace('-', '_').replace(' ', '_') for v in variables}
    safe_vars  = [rename_map[v] for v in variables]
    df_model   = df_model.rename(columns=rename_map)

    X       = df_model[safe_vars].values
    y_event = df_model['event'].astype(bool).values
    y_time  = df_model['stop'].values
    groups  = df_model['patient_id'].values
    y = np.array([(e, t) for e, t in zip(y_event, y_time)],
                 dtype=[('event', bool), ('time', float)])
    return X, y, y_event, groups

def run_cv(df, variables, n_splits=5, n_iterations=10):
    """
    StratifiedGroupKFold CV
    Returns: summary dict with mean, std, median, min, max,
             Q1, Q3, below_05, below_05_pct, n, scores
    """
    X, y, y_event, groups = prepare_data(df, variables)
    all_scores = {name: [] for name in ['LASSO-Cox', 'RSF', 'GBM']}

    for iteration in range(n_iterations):
        cv = StratifiedGroupKFold(
            n_splits=n_splits, shuffle=True,
            random_state=42 + iteration)
        for _, (train_idx, test_idx) in enumerate(
                cv.split(X, y_event, groups)):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            if y_test['event'].sum() == 0:
                continue

            scaler       = StandardScaler()
            X_train_sc   = scaler.fit_transform(X_train)
            X_test_sc    = scaler.transform(X_test)

            for model_name in ['LASSO-Cox', 'RSF', 'GBM']:
                try:
                    model = get_models()[model_name]
                    model.fit(X_train_sc, y_train)
                    pred  = model.predict(X_test_sc)
                    ci    = concordance_index_censored(
                        y_test['event'], y_test['time'], pred)[0]
                    all_scores[model_name].append(ci)
                except Exception:
                    continue

    summary = {}
    for model_name, scores in all_scores.items():
        if len(scores) > 0:
            arr = np.array(scores)
            summary[model_name] = {
                'mean':         round(float(np.mean(arr)), 4),
                'std':          round(float(np.std(arr)), 4),
                'median':       round(float(np.median(arr)), 4),
                'min':          round(float(np.min(arr)), 4),
                'max':          round(float(np.max(arr)), 4),
                'q1':           round(float(np.percentile(arr, 25)), 4),
                'q3':           round(float(np.percentile(arr, 75)), 4),
                'below_05':     int((arr < 0.5).sum()),
                'below_05_pct': round(float((arr < 0.5).mean() * 100), 1),
                'n':            len(scores),
                'scores':       scores
            }
    return summary

def result_row(vars_list, res, added_var=None, prev_avg=None):
    avg_c  = round(np.mean([r['mean'] for r in res.values()]), 4)
    best_m = max(['LASSO-Cox', 'RSF', 'GBM'], key=lambda m: res[m]['mean'])
    row = {
        'N_vars':          len(vars_list),
        'Added_Variable':  added_var if added_var else 'core',
        'Variables':       ' + '.join(vars_list),
        'LASSO_mean':      res['LASSO-Cox']['mean'],
        'LASSO_std':       res['LASSO-Cox']['std'],
        'LASSO_median':    res['LASSO-Cox']['median'],
        'LASSO_min':       res['LASSO-Cox']['min'],
        'LASSO_max':       res['LASSO-Cox']['max'],
        'LASSO_below05':   res['LASSO-Cox']['below_05'],
        'LASSO_below05pct':res['LASSO-Cox']['below_05_pct'],
        'RSF_mean':        res['RSF']['mean'],
        'RSF_std':         res['RSF']['std'],
        'RSF_median':      res['RSF']['median'],
        'RSF_min':         res['RSF']['min'],
        'RSF_max':         res['RSF']['max'],
        'RSF_below05':     res['RSF']['below_05'],
        'RSF_below05pct':  res['RSF']['below_05_pct'],
        'GBM_mean':        res['GBM']['mean'],
        'GBM_std':         res['GBM']['std'],
        'GBM_median':      res['GBM']['median'],
        'GBM_min':         res['GBM']['min'],
        'GBM_max':         res['GBM']['max'],
        'GBM_below05':     res['GBM']['below_05'],
        'GBM_below05pct':  res['GBM']['below_05_pct'],
        'Avg_C_index':     avg_c,
        'Delta':           round(avg_c - prev_avg, 4) if prev_avg else '-',
        'Best_model':      best_m,
        'Best_mean':       res[best_m]['mean'],
        'Best_std':        res[best_m]['std'],
    }
    return row, avg_c

# -----------------------------------------------------------------------------
# 2변수 Core
# -----------------------------------------------------------------------------
print("\n[2변수 Core]")
res_core = run_cv(df_cp, CORE_VARS)
row_core, avg_2var = result_row(CORE_VARS, res_core, 'core', None)
row_core['Delta'] = '-'
print(f"  LASSO={res_core['LASSO-Cox']['mean']:.4f} "
      f"RSF={res_core['RSF']['mean']:.4f} "
      f"GBM={res_core['GBM']['mean']:.4f} | avg={avg_2var:.4f}")

# -----------------------------------------------------------------------------
# 3변수 (Core + 1개씩)
# -----------------------------------------------------------------------------
print("\n[3변수 조합]")
three_rows = []
for add_var in ADDITIONAL_VARS:
    vars_i = CORE_VARS + [add_var]
    res_i  = run_cv(df_cp, vars_i)
    row_i, avg_i = result_row(vars_i, res_i, add_var, avg_2var)
    three_rows.append(row_i)
    direction = '+' if avg_i > avg_2var else '-'
    print(f"  + {add_var:<28} avg={avg_i:.4f} ({direction}{abs(avg_i-avg_2var):.4f})")

three_df  = pd.DataFrame(three_rows).sort_values('Avg_C_index', ascending=False)
best_3rd  = three_df.iloc[0]['Added_Variable']
best_3var = CORE_VARS + [best_3rd]
avg_3var  = three_df.iloc[0]['Avg_C_index']
print(f"\n  Best 3변수: {best_3var}  avg={avg_3var:.4f}")

# -----------------------------------------------------------------------------
# 4변수 (Best 3 + 1개씩)
# -----------------------------------------------------------------------------
print("\n[4변수 조합]")
remaining_4 = [v for v in ADDITIONAL_VARS if v != best_3rd]
four_rows   = []
for add_var in remaining_4:
    vars_i = best_3var + [add_var]
    res_i  = run_cv(df_cp, vars_i)
    row_i, avg_i = result_row(vars_i, res_i, add_var, avg_3var)
    four_rows.append(row_i)
    direction = '+' if avg_i > avg_3var else '-'
    print(f"  + {add_var:<28} avg={avg_i:.4f} ({direction}{abs(avg_i-avg_3var):.4f})")

four_df   = pd.DataFrame(four_rows).sort_values('Avg_C_index', ascending=False)
best_4th  = four_df.iloc[0]['Added_Variable']
best_4var = best_3var + [best_4th]
avg_4var  = four_df.iloc[0]['Avg_C_index']
print(f"\n  Best 4변수: {best_4var}  avg={avg_4var:.4f}")

# -----------------------------------------------------------------------------
# 5변수 (Best 4 + 1개씩)
# -----------------------------------------------------------------------------
print("\n[5변수 조합]")
remaining_5 = [v for v in remaining_4 if v != best_4th]
five_rows   = []
for add_var in remaining_5:
    vars_i = best_4var + [add_var]
    res_i  = run_cv(df_cp, vars_i)
    row_i, avg_i = result_row(vars_i, res_i, add_var, avg_4var)
    five_rows.append(row_i)
    direction = '+' if avg_i > avg_4var else '-'
    print(f"  + {add_var:<28} avg={avg_i:.4f} ({direction}{abs(avg_i-avg_4var):.4f})")

five_df   = pd.DataFrame(five_rows).sort_values('Avg_C_index', ascending=False)
best_5th  = five_df.iloc[0]['Added_Variable']
best_5var = best_4var + [best_5th]
avg_5var  = five_df.iloc[0]['Avg_C_index']
print(f"\n  Best 5변수: {best_5var}  avg={avg_5var:.4f}")

# -----------------------------------------------------------------------------
# 6변수 (Best 5 + 1개씩) - Overfitting Check
# -----------------------------------------------------------------------------
print("\n[6변수 조합 - Overfitting Check]")
remaining_6 = [v for v in remaining_5 if v != best_5th]
six_rows    = []
for add_var in remaining_6:
    vars_i = best_5var + [add_var]
    res_i  = run_cv(df_cp, vars_i)
    row_i, avg_i = result_row(vars_i, res_i, add_var, avg_5var)
    six_rows.append(row_i)
    direction = '+' if avg_i > avg_5var else '-'
    print(f"  + {add_var:<28} avg={avg_i:.4f} ({direction}{abs(avg_i-avg_5var):.4f})")

six_df    = pd.DataFrame(six_rows).sort_values('Avg_C_index', ascending=False)
best_6th  = six_df.iloc[0]['Added_Variable']
best_6var = best_5var + [best_6th]
avg_6var  = six_df.iloc[0]['Avg_C_index']
print(f"\n  Best 6변수: avg={avg_6var:.4f}")

# -----------------------------------------------------------------------------
# Trajectory 요약
# -----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("TRAJECTORY SUMMARY")
print("=" * 70)

trajectory = [
    {'N_vars': 2, 'Variables': ' + '.join(CORE_VARS),
     'Avg_C_index': avg_2var, 'Delta': '-',
     'LASSO_mean': res_core['LASSO-Cox']['mean'],
     'LASSO_std':  res_core['LASSO-Cox']['std'],
     'RSF_mean':   res_core['RSF']['mean'],
     'RSF_std':    res_core['RSF']['std'],
     'GBM_mean':   res_core['GBM']['mean'],
     'GBM_std':    res_core['GBM']['std']},
]
for n, best_vars, avg_c, prev_avg, df_n in [
    (3, best_3var, avg_3var, avg_2var, three_df),
    (4, best_4var, avg_4var, avg_3var, four_df),
    (5, best_5var, avg_5var, avg_4var, five_df),
    (6, best_6var, avg_6var, avg_5var, six_df),
]:
    br = df_n[df_n['Added_Variable'] == best_vars[-1]].iloc[0]
    trajectory.append({
        'N_vars':      n,
        'Variables':   ' + '.join(best_vars),
        'Avg_C_index': avg_c,
        'Delta':       f"{avg_c - prev_avg:+.4f}",
        'LASSO_mean':  br['LASSO_mean'], 'LASSO_std': br['LASSO_std'],
        'RSF_mean':    br['RSF_mean'],   'RSF_std':   br['RSF_std'],
        'GBM_mean':    br['GBM_mean'],   'GBM_std':   br['GBM_std'],
    })

traj_df = pd.DataFrame(trajectory)
print(f"\n{'N':<5} {'Avg_CI':<10} {'Delta':<10} "
      f"{'LASSO':<18} {'RSF':<18} {'GBM'}")
print("-" * 75)
for _, row in traj_df.iterrows():
    print(f"  {int(row['N_vars']):<5} {row['Avg_C_index']:<10.4f} "
          f"{str(row['Delta']):<10} "
          f"{row['LASSO_mean']:.4f}+-{row['LASSO_std']:.4f}  "
          f"{row['RSF_mean']:.4f}+-{row['RSF_std']:.4f}  "
          f"{row['GBM_mean']:.4f}+-{row['GBM_std']:.4f}")

# 최적 모델 결정 (avg 기준, 6변수에서 하락하면 5변수)
optimal_vars = best_5var if avg_6var <= avg_5var else best_6var
optimal_avg  = avg_5var  if avg_6var <= avg_5var else avg_6var
print(f"\n  Optimal: {optimal_vars}")
print(f"  Avg C-index: {optimal_avg:.4f}")

# =============================================================================
# PART 5: TOP 3 FOLD-WISE ANALYSIS
# =============================================================================

print("\n" + "=" * 70)
print("PART 5: TOP 3 FOLD-WISE ANALYSIS")
print("=" * 70)

# 5변수 결과에서 GBM 기준 Top 3 추출
five_df_sorted = five_df.sort_values('GBM_mean', ascending=False).reset_index(drop=True)

TOP3_VARS = {}
for i in range(3):
    added = five_df_sorted.iloc[i]['Added_Variable']
    TOP3_VARS[f'Rank{i+1}'] = {
        'vars':  best_4var + [added],
        'label': ' + '.join(best_4var + [added])
    }

print("\nTop 3 조합 (GBM 기준):")
for rank, info in TOP3_VARS.items():
    row = five_df_sorted.iloc[list(TOP3_VARS.keys()).index(rank)]
    print(f"  {rank}: GBM={row['GBM_mean']:.4f}+-{row['GBM_std']:.4f}")
    print(f"         {info['vars']}")

# Top 3 재실행 (fold-wise scores 포함)
top3_results = {}
for rank, info in TOP3_VARS.items():
    print(f"\n  {rank} 실행 중...")
    res = run_cv(df_cp, info['vars'])
    top3_results[rank] = {'vars': info['vars'], 'results': res}
    for m, r in res.items():
        print(f"    {m:<12} {r['mean']:.4f}+-{r['std']:.4f} "
              f"median={r['median']:.4f} <0.5:{r['below_05_pct']}%")

# =============================================================================
# PART 6: SHAP ANALYSIS (Top 3 x RSF + GBM)
# =============================================================================

print("\n" + "=" * 70)
print("PART 6: SHAP ANALYSIS")
print("=" * 70)

VAR_LABELS = {
    'dls-peak-number':      'DLS peak number',
    'sex':                  'Sex (male)',
    'education':            'Education',
    'dls-peak-1-size':      'DLS peak 1 size',
    'dls-peak-2-pd':        'DLS peak 2 PD',
    'dls-peak-2-intensity': 'DLS peak 2 intensity',
    'tht-forming-rate':     'THT forming rate',
    'dls-peak-1-intensity': 'DLS peak 1 intensity',
    'dls-peak-2-size':      'DLS peak 2 size',
    'neurotoxicity':        'Neurotoxicity',
    'dls-peak-1-pd':        'DLS peak 1 PD',
    'tht-mfi':              'THT MFI',
    'disease duration':     'Disease duration',
    'age':                  'Age',
    'tht-tlag':             'THT T-lag',
    'tht-t20':              'THT T20',
    'tht-t50':              'THT T50',
}

def prepare_shap_data(df, variables):
    df_model      = df.copy()
    rename_map    = {v: v.replace('-', '_').replace(' ', '_') for v in variables}
    safe_vars     = [rename_map[v] for v in variables]
    df_model      = df_model.rename(columns=rename_map)
    X             = df_model[safe_vars].values
    y_event       = df_model['event'].astype(bool).values
    y_time        = df_model['stop'].values
    y             = np.array([(e, t) for e, t in zip(y_event, y_time)],
                             dtype=[('event', bool), ('time', float)])
    display_names = [VAR_LABELS.get(v, v) for v in variables]
    return X, y, display_names

def compute_shap(X, y, model_type, display_names, random_seed=42):
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    if model_type == 'RSF':
        model = RandomSurvivalForest(n_estimators=100, max_depth=3,
                                     random_state=random_seed, n_jobs=-1)
    else:
        model = GradientBoostingSurvivalAnalysis(n_estimators=100, max_depth=2,
                                                  learning_rate=0.1,
                                                  random_state=random_seed)
    model.fit(X_scaled, y)

    explainer   = shap.Explainer(model.predict, X_scaled,
                                  feature_names=display_names)
    shap_values = explainer(X_scaled)

    mean_abs    = np.abs(shap_values.values).mean(axis=0)
    imp_df      = pd.DataFrame({
        'Variable':  display_names,
        'Mean_SHAP': mean_abs,
        'Model':     model_type
    }).sort_values('Mean_SHAP', ascending=False).reset_index(drop=True)

    return shap_values, imp_df, X_scaled

all_shap  = {}
all_imps  = []

for rank, info in TOP3_VARS.items():
    print(f"\n  [{rank}]")
    X, y, display_names = prepare_shap_data(df_cp, info['vars'])

    for model_type in ['RSF', 'GBM']:
        print(f"    {model_type} 계산 중...")
        shap_vals, imp_df, X_sc = compute_shap(X, y, model_type, display_names)
        imp_df['Rank'] = rank

        all_shap[f'{rank}_{model_type}'] = {
            'shap_values':   shap_vals,
            'importance_df': imp_df,
            'X_scaled':      X_sc,
            'display_names': display_names,
            'rank':          rank,
            'model_type':    model_type,
            'vars':          info['vars']
        }
        all_imps.append(imp_df)

        print(f"      Feature importance:")
        for _, row in imp_df.iterrows():
            print(f"        {row['Variable']:<30} {row['Mean_SHAP']:.4f}")

# SHAP Figure (PDF)
print("\n  SHAP Figure 생성 중...")
with PdfPages('/content/SHAP_Analysis_Top3.pdf') as pdf:
    for rank, info in TOP3_VARS.items():
        for model_type in ['RSF', 'GBM']:
            key = f'{rank}_{model_type}'
            res = all_shap[key]

            # Beeswarm
            fig = plt.figure(figsize=(10, 6))
            shap.summary_plot(res['shap_values'], res['X_scaled'],
                              feature_names=res['display_names'],
                              show=False, plot_type='dot')
            plt.title(f'SHAP Summary - {rank} ({model_type})\n{info["label"]}',
                      fontsize=11, fontweight='bold', pad=12)
            plt.tight_layout()
            pdf.savefig(bbox_inches='tight')
            plt.close()

            # Bar
            fig, ax = plt.subplots(figsize=(9, 5))
            imp = res['importance_df'].sort_values('Mean_SHAP', ascending=True)
            colors = ['#FF8C00' if v in ['Sex (male)', 'Education']
                      else '#2878B5' for v in imp['Variable']]
            bars = ax.barh(imp['Variable'], imp['Mean_SHAP'],
                           color=colors, edgecolor='white', height=0.6)
            for bar, val in zip(bars, imp['Mean_SHAP']):
                ax.text(bar.get_width() + 0.001,
                        bar.get_y() + bar.get_height() / 2,
                        f'{val:.4f}', va='center', ha='left', fontsize=10)
            ax.set_xlabel('Mean |SHAP value|', fontsize=12)
            ax.set_title(f'Feature Importance - {rank} ({model_type})\n{info["label"]}',
                         fontsize=11, fontweight='bold')
            ax.grid(axis='x', alpha=0.3)
            ax.set_xlim(0, imp['Mean_SHAP'].max() * 1.25)
            plt.tight_layout()
            pdf.savefig(bbox_inches='tight')
            plt.close()

print("  저장: SHAP_Analysis_Top3.pdf")

# =============================================================================
# PART 7: 전체 결과 저장
# =============================================================================

print("\n" + "=" * 70)
print("PART 7: 전체 결과 저장")
print("=" * 70)

# File 1: Forward_Selection_Complete.xlsx
with pd.ExcelWriter('/content/Forward_Selection_Complete.xlsx',
                    engine='openpyxl') as writer:

    # Trajectory
    traj_df.to_excel(writer, sheet_name='Trajectory', index=False)

    # Core 2변수
    pd.DataFrame([row_core]).to_excel(
        writer, sheet_name='2var_Core', index=False)

    # 3~6변수
    three_df.to_excel(writer, sheet_name='3var_Combinations', index=False)
    four_df.to_excel(writer,  sheet_name='4var_Combinations', index=False)
    five_df.to_excel(writer,  sheet_name='5var_Combinations', index=False)
    six_df.to_excel(writer,   sheet_name='6var_Combinations', index=False)

print("  저장: Forward_Selection_Complete.xlsx")

# File 2: Top3_Foldwise_Results.xlsx
with pd.ExcelWriter('/content/Top3_Foldwise_Results.xlsx',
                    engine='openpyxl') as writer:

    # Top3 Summary
    summary_rows = []
    for rank, data in top3_results.items():
        for m, r in data['results'].items():
            summary_rows.append({
                'Rank': rank, 'Variables': ' + '.join(data['vars']),
                'Model': m, 'Mean': r['mean'], 'SD': r['std'],
                'Median': r['median'], 'Min': r['min'], 'Max': r['max'],
                'Q1': r['q1'], 'Q3': r['q3'],
                'Below_05': r['below_05'], 'Below_05_pct': r['below_05_pct'],
            })
    pd.DataFrame(summary_rows).to_excel(
        writer, sheet_name='Top3_Summary', index=False)

    # Rank별 Fold-wise
    for rank, data in top3_results.items():
        foldwise_rows = []
        for m, r in data['results'].items():
            for i, score in enumerate(r['scores']):
                foldwise_rows.append({
                    'Rank': rank, 'Model': m,
                    'Eval_num': i + 1,
                    'C_index': round(score, 4),
                    'Below_05': 1 if score < 0.5 else 0
                })
        pd.DataFrame(foldwise_rows).to_excel(
            writer, sheet_name=f'{rank}_Foldwise', index=False)

    # GBM만 별도
    gbm_rows = []
    for rank, data in top3_results.items():
        r = data['results']['GBM']
        arr = np.array(r['scores'])
        for i, score in enumerate(r['scores']):
            gbm_rows.append({
                'Rank': rank, 'Variables': ' + '.join(data['vars']),
                'Eval_num': i + 1,
                'C_index': round(score, 4),
                'Below_05': 1 if score < 0.5 else 0
            })
    pd.DataFrame(gbm_rows).to_excel(
        writer, sheet_name='GBM_Foldwise', index=False)

    # GBM Stats Summary
    gbm_stats = []
    for rank, data in top3_results.items():
        r   = data['results']['GBM']
        arr = np.array(r['scores'])
        gbm_stats.append({
            'Rank': rank, 'Variables': ' + '.join(data['vars']),
            'Mean': r['mean'], 'SD': r['std'], 'Median': r['median'],
            'Min': r['min'], 'Max': r['max'], 'Q1': r['q1'], 'Q3': r['q3'],
            'Below_05_n': r['below_05'], 'Below_05_pct': r['below_05_pct'],
            'Above_09_n': int((arr >= 0.9).sum()),
            'Above_09_pct': round(float((arr >= 0.9).mean() * 100), 1),
            'Perfect_10_n': int((arr == 1.0).sum()),
            'Perfect_10_pct': round(float((arr == 1.0).mean() * 100), 1),
        })
    pd.DataFrame(gbm_stats).to_excel(
        writer, sheet_name='GBM_Stats_Summary', index=False)

print("  저장: Top3_Foldwise_Results.xlsx")

# File 3: SHAP_Results_Top3.xlsx
with pd.ExcelWriter('/content/SHAP_Results_Top3.xlsx',
                    engine='openpyxl') as writer:

    # Feature Importance 전체
    all_imp_df = pd.concat(all_imps, ignore_index=True)
    all_imp_df.to_excel(writer, sheet_name='Feature_Importance_All', index=False)

    # 각 Rank x Model SHAP values
    for rank, info in TOP3_VARS.items():
        for model_type in ['RSF', 'GBM']:
            key = f'{rank}_{model_type}'
            res = all_shap[key]
            shap_df = pd.DataFrame(res['shap_values'].values,
                                   columns=res['display_names'])
            shap_df.insert(0, 'patient_id', df_cp['patient_id'].values)
            shap_df.insert(1, 'event',      df_cp['event'].values)
            shap_df.to_excel(writer,
                             sheet_name=f'{rank}_{model_type}_SHAP',
                             index=False)

    # GBM Importance 비교
    gbm_imp   = all_imp_df[all_imp_df['Model'] == 'GBM'].copy()
    gbm_pivot = gbm_imp.pivot_table(
        index='Variable', columns='Rank',
        values='Mean_SHAP', aggfunc='first').reset_index()
    gbm_pivot.columns.name = None
    exist_ranks = [c for c in ['Rank1', 'Rank2', 'Rank3']
                   if c in gbm_pivot.columns]
    gbm_pivot['Avg_SHAP'] = gbm_pivot[exist_ranks].mean(axis=1)
    gbm_pivot = gbm_pivot.sort_values('Avg_SHAP', ascending=False)
    gbm_pivot.to_excel(writer,
                       sheet_name='GBM_Importance_Comparison', index=False)

print("  저장: SHAP_Results_Top3.xlsx")

# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)

print(f"""
KEY FINDINGS:
  Core model    (2 vars): avg C-index = {avg_2var:.4f}
  3-var model             avg C-index = {avg_3var:.4f}
  4-var model             avg C-index = {avg_4var:.4f}
  5-var model             avg C-index = {avg_5var:.4f}  <- OPTIMAL
  6-var model             avg C-index = {avg_6var:.4f}  (overfitting)

OPTIMAL MODEL:
  Variables: {optimal_vars}

TOP 3 (GBM):""")

for rank, data in top3_results.items():
    r = data['results']['GBM']
    print(f"  {rank}: {data['vars'][-1]:<28} "
          f"GBM={r['mean']:.4f}+-{r['std']:.4f} "
          f"median={r['median']:.4f}")

print(f"""
CV: {N_SPLITS}-fold x {N_ITERATIONS} iterations (StratifiedGroupKFold)

OUTPUT FILES:
  counting_process_final_v2.csv
  Univariate_Cox_Results_v2.xlsx
  Figure_Forest_Plot_v2.pdf / .png
  Forward_Selection_Complete.xlsx
  Top3_Foldwise_Results.xlsx
  SHAP_Results_Top3.xlsx
  SHAP_Analysis_Top3.pdf
""")